# Case-Study ESDA Core Draft

Runs the BG-first ESDA implementation draft from the validated Python notebook script. The notebook loads the audited block-group surface, previews the primary outcome and covariates, then writes the core correlation, Moran, and LISA outputs.

In [ ]:
from __future__ import annotations

import importlib.util
from pathlib import Path


def resolve_script_path(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        script_path = candidate / 'notebooks' / 'tabular' / '16_case_study_esda_core.py'
        if script_path.exists():
            return script_path
    raise FileNotFoundError('Could not find notebooks/tabular/16_case_study_esda_core.py from the current working directory.')


SCRIPT_PATH = resolve_script_path()
SPEC = importlib.util.spec_from_file_location('case_study_esda_core', SCRIPT_PATH)
case_study_esda_core = importlib.util.module_from_spec(SPEC)
SPEC.loader.exec_module(case_study_esda_core)
print({
    'script_path': str(SCRIPT_PATH),
    'bg_agg_table': case_study_esda_core.BG_AGG_TABLE,
    'correlation_output_path': str(case_study_esda_core.CORRELATION_OUTPUT_PATH),
})

## Load BG Analysis Surface

This stays on the block-group aggregate as the core analysis unit. The primary outcome is resolved from the available BG columns rather than hard-coded to a single stale surface.

In [ ]:
con = case_study_esda_core.connect(case_study_esda_core.resolve_db_path())
bg_surface = case_study_esda_core.load_bg_analysis_surface(con)
con.close()

outcome_column = case_study_esda_core.resolve_primary_outcome(bg_surface)
covariates = case_study_esda_core.resolve_analysis_covariates(bg_surface)
print(f"BG rows: {len(bg_surface):,}")
print(f"primary outcome: {outcome_column}")
print(f"covariates: {', '.join(covariates) if covariates else 'none'}")
preview_columns = [column_name for column_name in ('bg_geoid', 'municipio', outcome_column, *covariates[:4]) if column_name in bg_surface.columns]
bg_surface[preview_columns].head(10)

## Run Core ESDA Outputs

This writes the main correlation table, the per-municipality Moran outputs, the Queen-vs-Rook sensitivity table, and the primary-outcome LISA export and map.

In [ ]:
case_study_esda_core.REPORT_DIR.mkdir(parents=True, exist_ok=True)
case_study_esda_core.MAP_DIR.mkdir(parents=True, exist_ok=True)

correlations = case_study_esda_core.build_correlation_table(bg_surface, outcome_column=outcome_column, covariates=covariates)
correlations.to_csv(case_study_esda_core.CORRELATION_OUTPUT_PATH, index=False)

moran_frames = [case_study_esda_core.run_global_moran(bg_surface, value_column=outcome_column, weight_kind='queen')]
weight_sensitivity = case_study_esda_core.pd.concat([
    case_study_esda_core.run_global_moran(bg_surface, value_column=outcome_column, weight_kind='queen'),
    case_study_esda_core.run_global_moran(bg_surface, value_column=outcome_column, weight_kind='rook'),
], ignore_index=True)
for metric_name in [column_name for column_name in case_study_esda_core.SENSITIVITY_OUTCOME_CANDIDATES if column_name in bg_surface.columns]:
    moran_frames.append(case_study_esda_core.run_global_moran(bg_surface, value_column=metric_name, weight_kind='queen'))
morans_i = case_study_esda_core.pd.concat([frame for frame in moran_frames if not frame.empty], ignore_index=True) if any(not frame.empty for frame in moran_frames) else case_study_esda_core.pd.DataFrame()
if not morans_i.empty:
    morans_i.to_csv(case_study_esda_core.MORAN_OUTPUT_PATH, index=False)
weight_sensitivity.to_csv(case_study_esda_core.WEIGHT_SENSITIVITY_OUTPUT_PATH, index=False)

lisa_gdf = case_study_esda_core.run_local_moran(bg_surface, value_column=outcome_column)
if not lisa_gdf.empty:
    lisa_gdf.to_parquet(case_study_esda_core.LISA_OUTPUT_PATH, index=False)
    case_study_esda_core.plot_lisa_map(lisa_gdf, value_column=outcome_column, output_path=case_study_esda_core.LISA_MAP_OUTPUT_PATH)

print(f"correlations csv: {case_study_esda_core.CORRELATION_OUTPUT_PATH}")
print(f"weight sensitivity csv: {case_study_esda_core.WEIGHT_SENSITIVITY_OUTPUT_PATH}")
if not morans_i.empty:
    print(f"Moran's I csv: {case_study_esda_core.MORAN_OUTPUT_PATH}")
if not lisa_gdf.empty:
    print(f"LISA parquet: {case_study_esda_core.LISA_OUTPUT_PATH}")
    print(f"LISA map: {case_study_esda_core.LISA_MAP_OUTPUT_PATH}")
morans_i if not morans_i.empty else correlations